In [6]:
import numpy as np
from pathlib import Path
import os, sys
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
import warnings
import pandas as pd

repo_root = Path("/home/thardy/elefanto/ConsciousnessTeam_Data/SOUNDMODEL/Data_SoundGOOD/LEAD_ExperimentalFolder")
os.chdir(repo_root)

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))
os.environ["PYTHONPATH"] = str(repo_root) + os.pathsep + os.environ.get("PYTHONPATH", "")

import LEAD as lead


# Important variables
cwd = Path.cwd()
SNRs = np.array([-np.inf,-13,-11,-9,-7,-5,-3])
SubIDs = ['01','02','03','05','06','07','08','09','11','12','13','14','15','17','19','20','22','23','24','25']
colormap = {0: (0, 0, 0), 1: (0, 0.25, 1), 2: (0, 0.9375, 1), 3: (0, 0.91, 0.1), 4: (1, 0.6, 0), 5: (1, 0, 0), 6: (0.8, 0, 0)}

In [7]:
def count_sign_changes(delta_x):
    signs = np.sign(delta_x)
    sign_changes = np.sum(np.diff(signs) != 0)
    return sign_changes

def is_bifurcation(delta_x_list):
    counts = [count_sign_changes(delta_x) for delta_x in delta_x_list]
    unique_counts = set(counts)
    return len(unique_counts) > 1


# Bifurcation Probability via Particle Density Estimation

In [ ]:
task = "Passive"
period = 'early'
file_path = Path(f'Fit_{task}_{period}')

# Define the summary path clearly and load existing progress
summary_path = file_path / "BifurcationProbability" / "bifurcation_probabilities.csv"
if summary_path.exists():
    print(f"Found existing summary at {summary_path}. Loading progress...")
    existing_df = pd.read_csv(summary_path, index_col=0)
    bifurcation_prob = existing_df["probability_of_bifurcation"].to_dict()
    print(bifurcation_prob)
else:
    bifurcation_prob = {}

    
for part in range(20):
    print(f'part {part} lets go')

    # --- SKIP if already processed --- 
    if part in bifurcation_prob:
        print(f">>> Part {part} already exists in CSV. Skipping...")
        continue

    # --- Load Data ---
    data_ref = f'myEpochs_{task}/Epoch_{SubIDs[part]}-epo.fif'
    epochs_file = cwd.parents[0] / data_ref
    
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", category=RuntimeWarning)
        state_train = lead.STG(epochs_file, tmin=100, tmax=200)
        n_categories = len(list(state_train.keys()))
        categories = list(range(n_categories))

    # --- Input Definition ---
    one_input = np.concatenate((np.zeros(60), np.ones(10), np.zeros(180)))
    input_train = {cat: np.stack([one_input for _ in range(state_train[cat].shape[0])]) for cat in categories}

    # --- Data Reformating ---
    state_train_constant_stim = {cat: state_train[cat][:,60:70] for cat in range(1, n_categories)}
    input_train_constant_stim = {cat: input_train[cat][:,60:70] for cat in range(1, n_categories)}
    no_stim_sliced = np.concatenate([state_train[0][:, i*50:(i+1)*50] for i in range(5)])
    pre_stim_sliced = np.concatenate([state_train[cat][:, :50] for cat in range(1, n_categories)])
    state_train_constant_stim[0] = np.concatenate((no_stim_sliced, pre_stim_sliced))
    input_train_constant_stim[0] = np.zeros_like(state_train_constant_stim[0])
    
    # --- Load Parent Parameters ---
    gainfixed_parent = lead.model.StratifiedNonLinear1(
        tau=10, process_noise=0.1, measure_noise=0.1, threshold=1, sharpness=5, gain=0)
    param_path = file_path / f"GainFixedModel_part{part}_params"
    gainfixed_parent.load_params(param_path)

    # --- Define Ranges ---
    thresh_range = np.linspace(0, 2, 10)
    if gainfixed_parent.tau < 3:
        tau_range = np.linspace(1.5, 4.5, 10)
    else:
        tau_range = np.linspace(gainfixed_parent.tau - 1.5, gainfixed_parent.tau + 1.5, 10)

    # --- Grid Search for Tau & threshold at the same time (model StratifiedNonLinear1) ---
    ll_array, bif_array = np.zeros((len(tau_range), len(thresh_range))), np.zeros((len(tau_range), len(thresh_range)))
    for idx_tau, tau in tqdm(enumerate(tau_range), total=len(tau_range), desc=f"Grid Search for Part {part}"):
        for idx_th, threshold in enumerate(thresh_range):

            # --- Define Model Instance ---
            gainfixed = lead.model.StratifiedNonLinear1(
                tau=tau, 
                process_noise=gainfixed_parent.process_noise, 
                measure_noise=gainfixed_parent.measure_noise, 
                threshold=threshold, 
                gain=gainfixed_parent.gain,
                sharpness=5
            )
            gainfixed.set_params({f'w{cat}': getattr(gainfixed_parent, f'w{cat}') for cat in range(1, n_categories)})

            # A. Fit Relevant Parameters (strategy like lead.fitting_tools)
            for k_loop in range(2):
                
                # refine gain first
                gainfixed.fit(
                    state_series=state_train_constant_stim,
                    input_series=input_train_constant_stim,
                    init_params=[getattr(gainfixed, pname) for pname in gainfixed._param_names],
                    bounds=[(1,25), (0.01, 1), (0.01, 1)] + [(0, 1)]*7 + [(0, 0.5), (0, 2), (0, 10)],
                    fixed_params=['tau', 'process_noise', 'measure_noise', 'threshold', 'sharpness'] + [f'w{cat}' for cat in range(1, n_categories)])
                
                # refine the input weights
                for cat in range(1, n_categories):
                    
                    gainfixed_one_cat = lead.model.NonLinear1(
                        tau=gainfixed.tau, 
                        process_noise=gainfixed.process_noise, 
                        measure_noise=gainfixed.measure_noise,
                        input_weight=getattr(gainfixed, f'w{cat}'),
                        gain=gainfixed.gain,
                        threshold=gainfixed.threshold, 
                        sharpness=5
                    )
                    
                    gainfixed_one_cat.fit(
                        state_series={0: state_train_constant_stim[cat]},
                        input_series={0: input_train_constant_stim[cat]},
                        init_params=[getattr(gainfixed_one_cat, pname) for pname in gainfixed_one_cat._param_names],
                        bounds=[(1,25), (0.01, 1), (0.01, 1), (0, 1), (0, 1), (0, 2), (0, 10)],
                        fixed_params=['tau', 'process_noise', 'measure_noise', 'threshold', 'gain', 'sharpness'],
                        feedback=False
                    )
                    gainfixed.set_params({f'w{cat}': gainfixed_one_cat.input_weight})

            ll_array[idx_tau, idx_th] = gainfixed.loglikelihood(state_train_constant_stim, input_train_constant_stim)

            # B. Assess bifurcation (Linear Interpolation)
            states = np.linspace(-1, 3, 500)
            delta_x_list = []
            
            for cat in range(n_categories - 1):
                # Initial state diff
                delta_x_list.append(gainfixed.core(state=states, input_value=1, signal_category=cat) - states)
                
                w1, w2 = getattr(gainfixed, f'w{cat}'), getattr(gainfixed, f'w{cat+1}')
                
                # Travel on the w1 -> w2 segment
                for t in np.linspace(0, 1, 100):
                    w = w1 + t*(w2-w1)
                    
                    interp_model = lead.model.NonLinear1(
                        tau=gainfixed.tau, 
                        process_noise=gainfixed.process_noise, 
                        measure_noise=gainfixed.measure_noise, 
                        threshold=gainfixed.threshold, 
                        sharpness=gainfixed.sharpness, 
                        input_weight=w, 
                        gain=gainfixed.gain
                    )
                    # Note: signal_category=0 is used because NonLinear1 is non-stratified (single cat)
                    delta_x_list.append(interp_model.core(state=states, input_value=1, signal_category=0) - states)
            
            bifbool = is_bifurcation(delta_x_list) 
            bif_array[idx_tau, idx_th] = int(bifbool)

    
    # --- Process Results ---
    likelihood = np.exp(ll_array - np.max(ll_array))

    # Save Results
    os.makedirs(file_path / "BifurcationProbability", exist_ok=True)
    np.savez(file_path/ "BifurcationProbability" / f'2D_grid_search_part{part}.npz', threshold_range=thresh_range, tau_range=tau_range, ll=ll_array, likelihood=likelihood, bifbool=bif_array)

    # Probability Calc (Weighted average over the 2D surface)
    proba = np.average(bif_array, weights=likelihood)
    bifurcation_prob[part] = proba
    print(f'Part {part} has a {proba:.4f} probability of bifurcation.')
    
    # Save summary CSV
    df = pd.DataFrame.from_dict(
        bifurcation_prob, orient="index", columns=["probability_of_bifurcation"]
    ).sort_index()
    df.to_csv(file_path / "BifurcationProbability" / "bifurcation_probabilities.csv")

    # --- 3. Plotting (2D Heatmap with Contours) ---
    plt.figure(figsize=(10, 8))
    
    # Create 2D Meshgrid for plotting
    # thresh_range (x-axis), tau_range (y-axis)
    X, Y = np.meshgrid(thresh_range, tau_range)
    
    # 1. Plot Heatmap of Likelihood
    # Using pcolormesh for the heatmap
    pcm = plt.pcolormesh(X, Y, likelihood, shading='auto', cmap='viridis')
    plt.colorbar(pcm, label='Relative Likelihood')
    
    # 2. Plot Contour of Bifurcation Boundary
    # We draw a line at 0.5 (since bif_array is 0 or 1) to show the border
    # colors='red' makes the bifurcation line distinct
    if np.any(bif_array) and not np.all(bif_array):
        CS = plt.contour(X, Y, bif_array, levels=[0.5], colors='red', linewidths=2)
        # Create a proxy artist for the legend
        from matplotlib.lines import Line2D
        proxy_line = Line2D([0], [0], color='red', linewidth=2, label='Bifurcation Boundary')
        plt.legend(handles=[proxy_line], loc='upper right')
    elif np.all(bif_array):
        plt.title(f"Part {part} (All Bifurcation)")
    else:
        plt.title(f"Part {part} (No Bifurcation)")

    plt.title(f'Parameter Space: Tau vs Threshold - Part {part}\nWeighted Bifurcation Prob: {proba:.3f}')
    plt.xlabel('Threshold')
    plt.ylabel('Tau')
    
    # Save plot
    plt.savefig(file_path / "BifurcationProbability" / f'2D_likelihood_part{part}_{task}.png')
    plt.close()

Found existing summary at Fit_Passive_early/BifurcationProbability/bifurcation_probabilities.csv. Loading progress...
{0: 0.0}
part 0 lets go
>>> Part 0 already exists in CSV. Skipping...
part 1 lets go


Grid Search for Part 1:   0%|          | 0/10 [00:00<?, ?it/s]